In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

In [6]:
# =========================
# 1) CSV 파일들이 들어있는 폴더 경로
# =========================
folder_path = Path(r"C:\Users\user\Desktop\Gibeom\1. HI Lab\0. Projects\0. On going\3. [Lead] ionic DC-TENG\0. 실험자료\9. Revision\260705_1차 UTM 실험-실패\30x30x0.5")

# =========================
# 2) 샘플 규격 입력
# =========================
# 인장률 계산용 초기 길이 / 게이지 길이
sample_length = 10.0   #[mm]
sample_area = 900 #[mm^2]

# 방법 B: 단면적을 직접 알고 있으면 위 계산 대신 아래처럼 직접 입력
# sample_area = 1.25  # mm^2

# =========================
# 3) 출력 폴더 이름
# =========================
output_folder_1 = folder_path / "01_Displacement_Force"
output_folder_2 = folder_path / "02_Strain_Stress"

output_folder_1.mkdir(exist_ok=True)
output_folder_2.mkdir(exist_ok=True)

print("입력 폴더:", folder_path)
print("출력 폴더 1:", output_folder_1)
print("출력 폴더 2:", output_folder_2)
print("단면적:", sample_area, "mm^2")

입력 폴더: C:\Users\user\Desktop\Gibeom\1. HI Lab\0. Projects\0. On going\3. [Lead] ionic DC-TENG\0. 실험자료\9. Revision\260705_1차 UTM 실험-실패\30x30x0.5
출력 폴더 1: C:\Users\user\Desktop\Gibeom\1. HI Lab\0. Projects\0. On going\3. [Lead] ionic DC-TENG\0. 실험자료\9. Revision\260705_1차 UTM 실험-실패\30x30x0.5\01_Displacement_Force
출력 폴더 2: C:\Users\user\Desktop\Gibeom\1. HI Lab\0. Projects\0. On going\3. [Lead] ionic DC-TENG\0. 실험자료\9. Revision\260705_1차 UTM 실험-실패\30x30x0.5\02_Strain_Stress
단면적: 900 mm^2


In [7]:
def read_csv_auto_encoding(file_path):
    """
    CSV 인코딩을 자동으로 시도해서 읽는 함수.
    한국어 장비 CSV는 보통 cp949인 경우가 많음.
    """
    encodings = ["utf-8-sig", "cp949", "euc-kr", "utf-8"]
    
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, header=None, encoding=enc)
            return df, enc
        except UnicodeDecodeError:
            continue
    
    raise UnicodeDecodeError(f"인코딩을 읽을 수 없습니다: {file_path}")

KGF_TO_N = 9.80665

csv_files = list(folder_path.glob("*.csv"))

print(f"처리할 CSV 파일 수: {len(csv_files)}")

for file_path in csv_files:
    print(f"\n처리 중: {file_path.name}")
    
    # CSV 읽기
    raw_df, used_encoding = read_csv_auto_encoding(file_path)
    
    # 1행: 컬럼명, 2행: 단위, 3행부터 데이터라고 가정
    header_row = raw_df.iloc[0]
    unit_row = raw_df.iloc[1]
    data_df = raw_df.iloc[2:].copy()
    
    # 3열: 하중, 4열: 변위
    load_col_index = 2
    displacement_col_index = 3
    
    load_unit = str(unit_row.iloc[load_col_index]).strip().lower()
    displacement_unit = str(unit_row.iloc[displacement_col_index]).strip().lower()
    
    # 숫자 변환
    load_raw = pd.to_numeric(data_df.iloc[:, load_col_index], errors="coerce")
    displacement_mm = pd.to_numeric(data_df.iloc[:, displacement_col_index], errors="coerce")
    
    # NaN 제거
    valid = load_raw.notna() & displacement_mm.notna()
    load_raw = load_raw[valid].reset_index(drop=True)
    displacement_mm = displacement_mm[valid].reset_index(drop=True)
    
    # 하중 단위 변환
    if "kgf" in load_unit:
        force_N = load_raw * KGF_TO_N
        print("하중 단위: kgf → N 변환")
    elif "n" in load_unit:
        force_N = load_raw
        print("하중 단위: N 그대로 사용")
    else:
        force_N = load_raw
        print(f"경고: 하중 단위를 명확히 판단하지 못했습니다. 단위 = {load_unit}, 값 그대로 사용")
    
    # =========================
    # 출력 1: 변위 - 하중
    # =========================
    df_disp_force = pd.DataFrame({
        "Displacement (mm)": displacement_mm,
        "Force (N)": force_N
    })
    
    output_name_1 = file_path.stem + "_displacement_force.csv"
    df_disp_force.to_csv(
        output_folder_1 / output_name_1,
        index=False,
        encoding="utf-8-sig"
    )
    
    # =========================
    # 출력 2: 인장률 - 응력
    # =========================
    # 인장률 [%]
    strain_percent = displacement_mm / sample_length * 100
    
    # 응력 [MPa]
    # N / mm^2 = MPa
    stress_MPa = force_N / sample_area
    
    df_strain_stress = pd.DataFrame({
        "Strain (%)": strain_percent,
        "Stress (MPa)": stress_MPa
    })
    
    output_name_2 = file_path.stem + "_strain_stress.csv"
    df_strain_stress.to_csv(
        output_folder_2 / output_name_2,
        index=False,
        encoding="utf-8-sig"
    )
    
    print(f"저장 완료 1: {output_name_1}")
    print(f"저장 완료 2: {output_name_2}")

print("\n전체 처리 완료")

처리할 CSV 파일 수: 19

처리 중: 1. FEP-1.csv
하중 단위: kgf → N 변환
저장 완료 1: 1. FEP-1_displacement_force.csv
저장 완료 2: 1. FEP-1_strain_stress.csv

처리 중: 1. FEP-2.csv
하중 단위: kgf → N 변환
저장 완료 1: 1. FEP-2_displacement_force.csv
저장 완료 2: 1. FEP-2_strain_stress.csv

처리 중: 1. FEP-3.csv
하중 단위: kgf → N 변환
저장 완료 1: 1. FEP-3_displacement_force.csv
저장 완료 2: 1. FEP-3_strain_stress.csv

처리 중: 1. FEP-4.csv
하중 단위: kgf → N 변환
저장 완료 1: 1. FEP-4_displacement_force.csv
저장 완료 2: 1. FEP-4_strain_stress.csv

처리 중: 1. Tape.csv
하중 단위: N 그대로 사용
저장 완료 1: 1. Tape_displacement_force.csv
저장 완료 2: 1. Tape_strain_stress.csv

처리 중: 2. PTFE-1.csv
하중 단위: kgf → N 변환
저장 완료 1: 2. PTFE-1_displacement_force.csv
저장 완료 2: 2. PTFE-1_strain_stress.csv

처리 중: 2. PTFE-2.csv
하중 단위: kgf → N 변환
저장 완료 1: 2. PTFE-2_displacement_force.csv
저장 완료 2: 2. PTFE-2_strain_stress.csv

처리 중: 2. PTFE-3.csv
하중 단위: kgf → N 변환
저장 완료 1: 2. PTFE-3_displacement_force.csv
저장 완료 2: 2. PTFE-3_strain_stress.csv

처리 중: 2. PTFE-4.csv
하중 단위: kgf → N 변환
저장 완료 1: 2. PTFE-4_d